In [0]:
from pyspark.sql import functions as F

# Full path of the raw file in the landing zone
source_path = "/Volumes/workspace/bronze/landing/transactions_2026_10_10.csv"

# Read the CSV into a DataFrame
df_raw = (
    spark.read
    .format("csv")                  # tells Spark the file format
    .option("header", True)         # first line contains column names
    .option("inferSchema", False)   # do NOT guess types: keep every column as text
    .load(source_path)
)

# Show the structure Spark found
df_raw.printSchema()

In [0]:
# Add lineage (traceability) columns: when the row was loaded and from which file
df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())        # load time
    .withColumn("_source_file", F.col("_metadata.file_path")) # origin file path
)

In [0]:
# Write the DataFrame as a managed Delta table in the bronze schema
(
    df_bronze.write
    .format("delta")                              # table format
    .mode("overwrite")                            # replace the table if it already exists
    .saveAsTable("workspace.bronze.transactions") # full address: catalog.schema.table
)

In [0]:
# Read the table back from the catalog (not from the file) and inspect it
df_check = spark.table("workspace.bronze.transactions")

print("Total rows:", df_check.count())
display(df_check.limit(10))